In [0]:
-- 1. ¿Qué clientes operaron consistentemente por encima del precio de mercado? ¿En qué instrumentos?
-- Definimos "consistentemente" como más del 70% de sus operaciones por encima del mercado

WITH transacciones_vs_mercado AS (
  SELECT 
    t.cliente_sk,
    i.simbolo_base,
    t.precio_transaccion AS precio_operado,
    t.precio_mercado,
    CASE 
      WHEN t.precio_transaccion > t.precio_mercado THEN 1 
      ELSE 0 
    END AS opera_sobre_mercado,
    t.tipo_transaccion
  FROM gold.fact_transacciones AS t
  JOIN gold.dim_instrumento AS i ON t.instrumento_sk = i.instrumento_sk
  WHERE precio_mercado IS NOT NULL
    AND LOWER(tipo_transaccion) = 'compra'  -- Solo compras son relevantes para este análisis
),
clientes_consistentes AS (
  SELECT 
    cliente_sk,
    simbolo_base,
    COUNT(*) AS total_operaciones,
    SUM(opera_sobre_mercado) AS operaciones_sobre_mercado,
    ROUND(SUM(opera_sobre_mercado) * 100.0 / COUNT(*), 2) AS porcentaje_sobre_mercado,
    ROUND(AVG(precio_operado - precio_mercado), 2) AS diferencia_promedio
  FROM transacciones_vs_mercado
  GROUP BY cliente_sk, simbolo_base
  HAVING COUNT(*) >= 5 
    AND SUM(opera_sobre_mercado) * 100.0 / COUNT(*) > 70  
)
SELECT 
  cliente_sk,
  simbolo_base,
  total_operaciones,
  operaciones_sobre_mercado,
  porcentaje_sobre_mercado,
  diferencia_promedio
FROM clientes_consistentes
ORDER BY porcentaje_sobre_mercado DESC, total_operaciones DESC

In [0]:
--¿Cuáles son los instrumentos con mayor desvío promedio entre precio operado y precio de mercado?
WITH desvios_por_instrumento AS (
  SELECT 
    c.instrumento_sk,
    c.simbolo_base,
    c.descripcion_titulo,
    c.tipo_instrumento,
    COUNT(*) AS total_transacciones,
    ROUND(AVG(ABS(t.precio_transaccion - t.precio_mercado)), 4) AS desvio_absoluto_promedio
  FROM gold.fact_transacciones t
  JOIN gold.dim_instrumento AS c 
  ON t.instrumento_sk = c.instrumento_sk
  WHERE t.precio_mercado IS NOT NULL
    AND t.precio_mercado > 0
  GROUP BY c.simbolo_base, c.descripcion_titulo, c.tipo_instrumento
  HAVING COUNT(*) >= 10  --Al menos 10 transacciones para tener significancia estadística
)
SELECT 
  instrumento_sk,
  simbolo_base,
  descripcion_titulo,
  tipo_instrumento,
  total_transacciones,
  desvio_absoluto_promedio
FROM desvios_por_instrumento
ORDER BY desvio_absoluto_promedio DESC
LIMIT 30

In [0]:
-- 3. ¿Cómo evolucionó la proporción de operaciones por canal mes a mes?

WITH operaciones_por_canal_mes AS (
  SELECT 
    f.anio AS anio,
    f.mes AS mes
    o.origen AS canal,
    COUNT(*) AS total_operaciones,
    SUM(cantidad * t.precio_transaccion) AS volumen_operado
  FROM gold.fact_transacciones as t
  JOIN gold.dim_origen as o ON fact_transacciones.origen_sk = dim_origen.origen_sk
  JOIN gold.dim_fecha AS f ON t.fecha_sk = f.fecha_sk
  GROUP BY anio,mes
),
total_por_mes AS (
  SELECT 
    mes,
    SUM(total_operaciones) AS operaciones_totales_mes,
    SUM(volumen_operado) AS volumen_total_mes
  FROM operaciones_por_canal_mes
  GROUP BY mes
)
SELECT 
  DATE_FORMAT(ocm.mes, 'yyyy-MM') AS anio_mes,
  ocm.canal,
  ocm.total_operaciones,
  tm.operaciones_totales_mes,
  ROUND(ocm.total_operaciones * 100.0 / tm.operaciones_totales_mes, 2) AS porcentaje_operaciones,
  ROUND(ocm.volumen_operado, 2) AS volumen_operado,
  ROUND(ocm.volumen_operado * 100.0 / tm.volumen_total_mes, 2) AS porcentaje_volumen
FROM operaciones_por_canal_mes ocm
INNER JOIN total_por_mes tm ON ocm.mes = tm.mes
ORDER BY ocm.mes, porcentaje_operaciones DESC

In [0]:
-- 4. De los clientes que operaron en enero, ¿qué porcentaje volvió a operar en febrero y en marzo?

WITH clientes_enero AS (
  SELECT DISTINCT id_cliente
  FROM silver.transacciones
  WHERE DATE_TRUNC('MONTH', fecha) = '2025-01-01'
),
clientes_febrero AS (
  SELECT DISTINCT id_cliente
  FROM silver.transacciones
  WHERE DATE_TRUNC('MONTH', fecha) = '2025-02-01'
),
clientes_marzo AS (
  SELECT DISTINCT id_cliente
  FROM silver.transacciones
  WHERE DATE_TRUNC('MONTH', fecha) = '2025-03-01'
),
analisis_retencion AS (
  SELECT 
    COUNT(DISTINCT ce.id_cliente) AS clientes_enero,
    COUNT(DISTINCT cf.id_cliente) AS clientes_enero_en_febrero,
    COUNT(DISTINCT cm.id_cliente) AS clientes_enero_en_marzo,
    COUNT(DISTINCT CASE WHEN cf.id_cliente IS NOT NULL AND cm.id_cliente IS NOT NULL THEN ce.id_cliente END) AS clientes_enero_en_ambos
  FROM clientes_enero ce
  LEFT JOIN clientes_febrero cf ON ce.id_cliente = cf.id_cliente
  LEFT JOIN clientes_marzo cm ON ce.id_cliente = cm.id_cliente
)
SELECT 
  clientes_enero AS total_clientes_enero,
  clientes_enero_en_febrero,
  ROUND(clientes_enero_en_febrero * 100.0 / clientes_enero, 2) AS porcentaje_retencion_febrero,
  clientes_enero_en_marzo,
  ROUND(clientes_enero_en_marzo * 100.0 / clientes_enero, 2) AS porcentaje_retencion_marzo,
  clientes_enero_en_ambos,
  ROUND(clientes_enero_en_ambos * 100.0 / clientes_enero, 2) AS porcentaje_retencion_ambos_meses
FROM analisis_retencion